# 🌳 Decision Trees — Complete Implementation Guide

> **A beginner-friendly, placement-ready notebook covering Decision Tree classification from scratch using the Iris dataset.**

---

## 📋 Notebook Contents

| # | Section |
|---|--------|
| 1 | Introduction |
| 2 | Import Libraries |
| 3 | Load Dataset |
| 4 | Exploratory Data Analysis |
| 5 | Data Visualization |
| 6 | Train-Test Split |
| 7 | Build Decision Tree Classifier |
| 8 | Model Training |
| 9 | Predictions |
| 10 | Accuracy Evaluation |
| 11 | Confusion Matrix |
| 12 | Precision |
| 13 | Recall |
| 14 | F1 Score |
| 15 | Decision Tree Visualization |
| 16 | Feature Importance |
| 17 | Hyperparameter Tuning |
| 18 | Compare Gini vs Entropy |
| 19 | Practical Observations |
| 20 | Conclusion |

---

## 1. Introduction

A **Decision Tree** is a supervised learning algorithm that splits data into subsets based on feature conditions, forming a tree-like structure. Each internal node represents a decision on a feature, each branch represents the outcome, and each leaf node represents a prediction.

**Why the Iris Dataset?**
- Clean, well-balanced, no missing values
- 4 numerical features → easy to visualize
- 3 classes → multi-class classification
- Industry-standard benchmark dataset

**What we will cover:**
- Data exploration and visualization
- Training a Decision Tree classifier
- Evaluating with accuracy, confusion matrix, precision, recall, F1
- Visualizing the tree and feature importances
- Hyperparameter tuning with GridSearchCV
- Comparing Gini vs Entropy criteria

---

## 2. Import Libraries

We import all required libraries upfront. This is a good practice — it makes dependencies clear and avoids scattered imports throughout the notebook.

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Dataset ──────────────────────────────────────────────────────
from sklearn.datasets import load_iris

# ── Preprocessing ────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

# ── Decision Tree Model ──────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

# ── Evaluation Metrics ───────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay
)

# ── Settings ─────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# Global plot style
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100
RANDOM_STATE = 42

print("✅ All libraries imported successfully!")
print(f"NumPy    : {np.__version__}")
print(f"Pandas   : {pd.__version__}")
print(f"Seaborn  : {sns.__version__}")

---

## 3. Load the Iris Dataset

The **Iris dataset** (Fisher, 1936) contains measurements of 150 iris flowers across 3 species:
- *Iris setosa*
- *Iris versicolor*
- *Iris virginica*

**Features (all in cm):**
1. Sepal Length
2. Sepal Width
3. Petal Length
4. Petal Width

**Target:** Species (0 = Setosa, 1 = Versicolor, 2 = Virginica)

In [ ]:
# Load dataset
iris = load_iris()

# Create a DataFrame for easier analysis
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map(dict(enumerate(iris.target_names)))

# Store references
X = iris.data
y = iris.target
feature_names = iris.feature_names
class_names   = iris.target_names

print(f"Dataset shape   : {df.shape}")
print(f"Features        : {feature_names.tolist()}")
print(f"Target classes  : {class_names.tolist()}")
print(f"Samples per class: {dict(zip(class_names, np.bincount(y)))}")
print("\nFirst 5 rows:")
df.head()

---

## 4. Exploratory Data Analysis (EDA)

Before building any model, it is essential to **understand the data**:
- Check shape, types, and missing values
- Compute summary statistics
- Understand class distribution

EDA helps identify potential issues (missing values, outliers, class imbalance) before they affect model performance.

In [ ]:
print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(f"  Shape            : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Missing values   : {df.isnull().sum().sum()}")
print(f"  Duplicate rows   : {df.duplicated().sum()}")
print(f"  Data types       :")
for col, dtype in df.dtypes.items():
    print(f"    {col:<35} {dtype}")
print("="*55)

In [ ]:
# Statistical summary
print("\nDescriptive Statistics (Features Only):")
df[feature_names].describe().round(3)

In [ ]:
# Class distribution
print("\nClass Distribution:")
class_dist = df['species_name'].value_counts()
for name, count in class_dist.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f"  {name:<15} {bar} {count} ({pct:.1f}%)")

print("\n💡 The dataset is perfectly balanced (50 samples per class).")
print("   No resampling needed — we can use standard metrics directly.")

In [ ]:
# Per-class feature statistics
print("\nPer-Class Mean Feature Values:")
df.groupby('species_name')[list(feature_names)].mean().round(3)

**Observations from EDA:**
- No missing values or duplicates → data is clean
- Perfectly balanced classes (50 each) → no class imbalance
- Petal length and petal width show large differences across species → likely strong predictors
- Sepal width has high overlap across classes → weaker discriminator

---

## 5. Data Visualization

Visualizations help us understand feature distributions, class separability, and correlations. These insights guide both feature selection and model interpretation.

In [ ]:
# ── Figure 1: Feature Distributions ─────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Feature Distributions by Species', fontsize=16, fontweight='bold', y=1.01)

colors = {'setosa': '#4CAF50', 'versicolor': '#2196F3', 'virginica': '#F44336'}

for ax, feature in zip(axes.flat, feature_names):
    for species, color in colors.items():
        data = df[df['species_name'] == species][feature]
        ax.hist(data, alpha=0.6, bins=15, color=color, label=species, edgecolor='white')
    ax.set_title(feature.replace(' (cm)', ''), fontsize=12, fontweight='bold')
    ax.set_xlabel('Value (cm)', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()
print("💡 Petal features show clear separation across species — they are strong predictors.")

In [ ]:
# ── Figure 2: Pairplot ───────────────────────────────────────────
print("Generating pairplot (may take a few seconds)...")
g = sns.pairplot(
    df,
    hue='species_name',
    palette=colors,
    diag_kind='kde',
    plot_kws={'alpha': 0.6, 's': 40},
    height=2.4
)
g.figure.suptitle('Pairplot — Iris Features by Species', y=1.02, fontsize=14, fontweight='bold')
plt.savefig('pairplot.png', dpi=100, bbox_inches='tight')
plt.show()
print("💡 Setosa is linearly separable from the other two species.")
print("   Versicolor and Virginica overlap — a deeper tree may be needed.")

In [ ]:
# ── Figure 3: Correlation Heatmap ───────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

corr = df[list(feature_names)].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    mask=mask,
    ax=ax,
    linewidths=0.5,
    square=True,
    vmin=-1, vmax=1,
    annot_kws={'size': 11}
)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=15)
# Fix tick label cutoff
short_names = ['Sepal Len', 'Sepal Wid', 'Petal Len', 'Petal Wid']
ax.set_xticklabels(short_names, rotation=30, ha='right')
ax.set_yticklabels(short_names, rotation=0)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print("💡 Petal Length and Petal Width are highly correlated (0.96).")
print("   Sepal Width is negatively correlated with petal features.")

In [ ]:
# ── Figure 4: Boxplot ────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Feature Spread by Species (Boxplot)', fontsize=14, fontweight='bold')

palette = {'setosa': '#4CAF50', 'versicolor': '#2196F3', 'virginica': '#F44336'}

for ax, feature in zip(axes, feature_names):
    sns.boxplot(
        data=df,
        x='species_name',
        y=feature,
        palette=palette,
        ax=ax,
        width=0.5,
        linewidth=1.5,
        flierprops=dict(marker='o', markerfacecolor='gray', markersize=4)
    )
    ax.set_title(feature.replace(' (cm)', ''), fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=15)

plt.tight_layout()
plt.savefig('boxplot.png', dpi=100, bbox_inches='tight')
plt.show()

---

## 6. Train-Test Split

We split the data into training and test sets:
- **Training set (80%):** Used to fit the model
- **Test set (20%):** Used to evaluate generalization

`stratify=y` ensures both splits maintain the same class distribution as the original dataset — important for balanced evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y          # Preserve class distribution
)

print(f"Training samples  : {X_train.shape[0]}  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test samples      : {X_test.shape[0]}  ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"Feature dimensions: {X_train.shape[1]}")
print()
print("Class distribution after split (stratified):")
for split_name, y_split in [("Train", y_train), ("Test", y_test)]:
    counts = np.bincount(y_split)
    print(f"  {split_name}: " + " | ".join([f"{class_names[i]}: {c}" for i, c in enumerate(counts)]))

---

## 7. Build Decision Tree Classifier

scikit-learn implements the **CART algorithm** (Classification and Regression Trees). It builds binary trees using the Gini index (default) or entropy as the splitting criterion.

**Key parameters used here:**
- `criterion='gini'`: Splitting criterion (Gini impurity)
- `max_depth=4`: Limit tree depth to prevent overfitting
- `random_state=42`: Ensures reproducibility

In [ ]:
# Initialize the Decision Tree Classifier
clf = DecisionTreeClassifier(
    criterion='gini',
    max_depth=4,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=RANDOM_STATE
)

print("Decision Tree Classifier initialized with parameters:")
print("-" * 45)
for param, val in clf.get_params().items():
    print(f"  {param:<25} : {val}")

---

## 8. Model Training

Training a Decision Tree means finding the best series of splits on the training data that minimize impurity. The `.fit()` call does all of this internally.

After fitting, we can inspect the tree structure: depth, number of nodes, and leaves.

In [ ]:
# Train the model
clf.fit(X_train, y_train)

print("✅ Model trained successfully!")
print()
print(f"Tree depth         : {clf.get_depth()}")
print(f"Total nodes        : {clf.tree_.node_count}")
print(f"Number of leaves   : {clf.get_n_leaves()}")
print(f"Number of features : {clf.n_features_in_}")
print(f"Number of classes  : {clf.n_classes_}")

In [ ]:
# Print text-based tree structure
print("\nDecision Tree Rules (Text Representation):")
print("-" * 55)
tree_rules = export_text(clf, feature_names=list(feature_names))
print(tree_rules)

---

## 9. Predictions

Once trained, we use the model to predict class labels and class probabilities for the test set.

- `predict()` returns the most likely class
- `predict_proba()` returns class probabilities for all classes

In [ ]:
# Class predictions
y_pred = clf.predict(X_test)

# Probability predictions
y_proba = clf.predict_proba(X_test)

print("Sample Predictions (first 15 test samples):")
print("-" * 60)
print(f"{'Index':<8} {'Actual':<18} {'Predicted':<18} {'Correct?'}")
print("-" * 60)
for i in range(15):
    actual    = class_names[y_test[i]]
    predicted = class_names[y_pred[i]]
    correct   = '✅' if y_test[i] == y_pred[i] else '❌'
    print(f"{i:<8} {actual:<18} {predicted:<18} {correct}")

print(f"\nProbability output shape: {y_proba.shape}  (samples × classes)")
print(f"Example probabilities for sample 0: {y_proba[0]}")
print(f"  → Predicted class: {class_names[np.argmax(y_proba[0])]} (confidence: {y_proba[0].max():.2f})")

---

## 10. Accuracy Evaluation

**Accuracy** = (Correct Predictions) / (Total Predictions)

It is the simplest metric but can be misleading for imbalanced datasets. Since the Iris dataset is balanced, accuracy is a reliable metric here.

We evaluate on both training and test sets to detect overfitting:
- If **train >> test accuracy** → overfitting
- If both are low → underfitting

In [ ]:
train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc  = accuracy_score(y_test, y_pred)
gap       = train_acc - test_acc

print("=" * 40)
print("  ACCURACY EVALUATION")
print("=" * 40)
print(f"  Training Accuracy  : {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"  Test Accuracy      : {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"  Accuracy Gap       : {gap:.4f}")
print("=" * 40)

if gap < 0.05:
    print("\n✅ Model generalizes well — small gap between train and test.")
elif gap < 0.10:
    print("\n⚠️  Slight overfitting — consider reducing max_depth.")
else:
    print("\n❌ Overfitting detected — apply pruning or increase min_samples_leaf.")

# Cross-validation for robust estimate
cv_scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
print(f"\n5-Fold Cross-Validation Accuracy:")
print(f"  Scores   : {[round(s, 4) for s in cv_scores]}")
print(f"  Mean     : {cv_scores.mean():.4f}")
print(f"  Std Dev  : {cv_scores.std():.4f}")

---

## 11. Confusion Matrix

The **Confusion Matrix** is a table that shows the counts of:
- **True Positives (TP):** Correctly predicted positive
- **True Negatives (TN):** Correctly predicted negative
- **False Positives (FP):** Predicted positive, actually negative (Type I Error)
- **False Negatives (FN):** Predicted negative, actually positive (Type II Error)

For multi-class problems, it shows the count of predictions for every pair of (actual, predicted) classes.

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix (raw counts):")
print("-" * 45)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
cm_df.index.name   = 'Actual'
cm_df.columns.name = 'Predicted'
print(cm_df)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp1.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)
disp2.plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_ylabel('True Label', fontsize=11)

# Format normalized matrix values
for text in axes[1].texts:
    text.set_text(f"{float(text.get_text()):.2f}")

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

---

## 12. Precision

**Precision** answers: *"Of all samples predicted as class X, how many actually belong to class X?"*

$$\text{Precision} = \frac{TP}{TP + FP}$$

High precision means fewer false positives — important in applications like spam detection (we don't want to mark legitimate emails as spam).

In [ ]:
precision_per_class = precision_score(y_test, y_pred, average=None)
precision_macro     = precision_score(y_test, y_pred, average='macro')
precision_weighted  = precision_score(y_test, y_pred, average='weighted')

print("Precision Scores:")
print("-" * 40)
for i, (cls, prec) in enumerate(zip(class_names, precision_per_class)):
    bar = '█' * int(prec * 20)
    print(f"  {cls:<15} {bar:<20} {prec:.4f}")

print(f"\n  Macro Average    : {precision_macro:.4f}")
print(f"  Weighted Average : {precision_weighted:.4f}")
print()
print("💡 Macro average treats all classes equally.")
print("   Weighted average accounts for class size.")

---

## 13. Recall

**Recall (Sensitivity)** answers: *"Of all actual class X samples, how many did we correctly identify?"*

$$\text{Recall} = \frac{TP}{TP + FN}$$

High recall means fewer false negatives — critical in medical diagnosis (we don't want to miss actual disease cases).

In [ ]:
recall_per_class = recall_score(y_test, y_pred, average=None)
recall_macro     = recall_score(y_test, y_pred, average='macro')
recall_weighted  = recall_score(y_test, y_pred, average='weighted')

print("Recall Scores:")
print("-" * 40)
for cls, rec in zip(class_names, recall_per_class):
    bar = '█' * int(rec * 20)
    print(f"  {cls:<15} {bar:<20} {rec:.4f}")

print(f"\n  Macro Average    : {recall_macro:.4f}")
print(f"  Weighted Average : {recall_weighted:.4f}")

---

## 14. F1 Score

**F1 Score** is the harmonic mean of Precision and Recall. It balances both metrics and is especially useful when classes are imbalanced.

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

An F1 of 1.0 is perfect; 0.0 is worst.

In [ ]:
f1_per_class = f1_score(y_test, y_pred, average=None)
f1_macro     = f1_score(y_test, y_pred, average='macro')
f1_weighted  = f1_score(y_test, y_pred, average='weighted')

print("F1 Scores:")
print("-" * 40)
for cls, f1 in zip(class_names, f1_per_class):
    bar = '█' * int(f1 * 20)
    print(f"  {cls:<15} {bar:<20} {f1:.4f}")

print(f"\n  Macro Average    : {f1_macro:.4f}")
print(f"  Weighted Average : {f1_weighted:.4f}")

# Full classification report
print("\n" + "=" * 60)
print("  FULL CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# ── Metrics Summary Barplot ──────────────────────────────────────
metrics_df = pd.DataFrame({
    'Class': list(class_names) * 3,
    'Metric': ['Precision'] * 3 + ['Recall'] * 3 + ['F1 Score'] * 3,
    'Score': list(precision_per_class) + list(recall_per_class) + list(f1_per_class)
})

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(
    data=metrics_df, x='Class', y='Score', hue='Metric',
    palette=['#4CAF50', '#2196F3', '#FF9800'], ax=ax
)
ax.set_title('Precision, Recall & F1 Score by Class', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=11)
ax.set_xlabel('Species', fontsize=11)
ax.legend(title='Metric', fontsize=10)

# Annotate bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('metrics_barplot.png', dpi=100, bbox_inches='tight')
plt.show()

---

## 15. Decision Tree Visualization

One of the greatest advantages of Decision Trees is that they can be **visually inspected**. Each node shows:
- The splitting feature and threshold
- Gini impurity at that node
- Number of samples reaching the node
- Class value counts
- Color intensity = purity (darker = more pure)

In [ ]:
# ── Decision Tree Plot ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 10))

plot_tree(
    clf,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,           # Color nodes by majority class
    rounded=True,          # Rounded boxes
    fontsize=10,
    ax=ax,
    impurity=True,         # Show Gini at each node
    proportion=False       # Show raw sample counts
)

ax.set_title(
    'Decision Tree — Iris Classification (max_depth=4, Gini)',
    fontsize=15, fontweight='bold', pad=20
)

plt.tight_layout()
plt.savefig('decision_tree.png', dpi=120, bbox_inches='tight')
plt.show()

print("\n🔍 How to read this tree:")
print("  • Each box = one node")
print("  • Feature ≤ threshold? → Left branch (True), Right branch (False)")
print("  • gini = impurity at that node (0 = pure)")
print("  • samples = training samples reaching this node")
print("  • value = [class0_count, class1_count, class2_count]")
print("  • class = majority class at this node")
print("  • Color intensity = purity (darker blue = more pure)")

---

## 16. Feature Importance

Feature importance in Decision Trees measures the **total Gini reduction** each feature contributes across all splits, normalized to sum to 1.0.

A high importance score means the feature was frequently used for splitting and led to significant purity improvements.

In [ ]:
importances = clf.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("Feature Importance Scores:")
print("-" * 50)
for _, row in feat_imp_df.iterrows():
    bar = '█' * int(row['Importance'] * 40)
    print(f"  {row['Feature']:<35} {bar:<40} {row['Importance']:.4f}")

print(f"\n  Total (should = 1.0): {importances.sum():.4f}")

In [ ]:
# ── Feature Importance Plot ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

colors_imp = ['#E53935' if i == 0 else '#42A5F5' for i in range(len(feat_imp_df))]
bars = ax.barh(
    feat_imp_df['Feature'],
    feat_imp_df['Importance'],
    color=colors_imp,
    edgecolor='white',
    height=0.55
)

# Annotate
for bar, val in zip(bars, feat_imp_df['Importance']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Feature Importance (Gini Reduction)', fontsize=11)
ax.set_title('Feature Importance — Decision Tree', fontsize=14, fontweight='bold')
ax.set_xlim(0, feat_imp_df['Importance'].max() + 0.08)
ax.invert_yaxis()

top = feat_imp_df.iloc[0]
ax.text(0.98, 0.05, f"Most Important:\n{top['Feature']}",
        transform=ax.transAxes, ha='right', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFECB3', alpha=0.8))

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n💡 '{top['Feature']}' is the most discriminative feature.")
print("   This aligns with our EDA — petal features clearly separate species.")

---

## 17. Hyperparameter Tuning

**Hyperparameter tuning** finds the optimal configuration for the model. We use **GridSearchCV** with 5-fold cross-validation.

Key hyperparameters we tune:
- `max_depth`: Controls tree complexity
- `min_samples_split`: Minimum samples to split a node
- `min_samples_leaf`: Minimum samples in each leaf
- `criterion`: Splitting metric (Gini or Entropy)

In [ ]:
param_grid = {
    'max_depth'         : [2, 3, 4, 5, 6, None],
    'min_samples_split' : [2, 5, 10],
    'min_samples_leaf'  : [1, 2, 4],
    'criterion'         : ['gini', 'entropy']
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
best_cv_score = grid_search.best_score_
best_estimator = grid_search.best_estimator_

print("GridSearchCV Results:")
print("-" * 45)
print(f"  Combinations tested : {len(grid_search.cv_results_['params'])}")
print(f"  Best CV Accuracy    : {best_cv_score:.4f} ({best_cv_score*100:.2f}%)")
print(f"\n  Best Parameters:")
for k, v in best_params.items():
    print(f"    {k:<25} : {v}")

In [ ]:
# Evaluate best model on test set
y_pred_best = best_estimator.predict(X_test)
test_acc_best = accuracy_score(y_test, y_pred_best)

print("Performance Comparison — Default vs Tuned:")
print("-" * 50)
print(f"  {'Metric':<30} {'Default':>10} {'Tuned':>10}")
print("-" * 50)
print(f"  {'Test Accuracy':<30} {test_acc:>10.4f} {test_acc_best:>10.4f}")
print(f"  {'F1 Score (macro)':<30} {f1_macro:>10.4f} {f1_score(y_test, y_pred_best, average='macro'):>10.4f}")
print(f"  {'Tree Depth':<30} {clf.get_depth():>10} {best_estimator.get_depth():>10}")
print(f"  {'Leaf Count':<30} {clf.get_n_leaves():>10} {best_estimator.get_n_leaves():>10}")

In [ ]:
# ── Effect of max_depth on Accuracy ─────────────────────────────
depths = list(range(1, 16))
train_accs, test_accs = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    m.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, m.predict(X_train)))
    test_accs.append(accuracy_score(y_test, m.predict(X_test)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_accs, 'o-', color='#E53935', linewidth=2, markersize=6, label='Training Accuracy')
ax.plot(depths, test_accs,  's-', color='#1E88E5', linewidth=2, markersize=6, label='Test Accuracy')

best_depth = depths[test_accs.index(max(test_accs))]
ax.axvline(x=best_depth, color='gray', linestyle='--', alpha=0.7)
ax.annotate(f'Best depth = {best_depth}',
            xy=(best_depth, max(test_accs)),
            xytext=(best_depth + 1, max(test_accs) - 0.03),
            fontsize=10, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray'))

ax.fill_between(depths, train_accs, test_accs, alpha=0.08, color='red')
ax.text(12, 0.88, 'Overfitting\nZone', color='red', fontsize=9, alpha=0.7)

ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Effect of max_depth on Training vs Test Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0.7, 1.05)
ax.set_xticks(depths)
plt.tight_layout()
plt.savefig('depth_vs_accuracy.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n💡 Optimal max_depth for this dataset: {best_depth}")
print("   Beyond this, training accuracy increases but test accuracy stagnates → overfitting.")

---

## 18. Compare Gini vs. Entropy

scikit-learn supports two splitting criteria:
- **Gini:** `1 - Σpᵢ²` — faster, no logarithm
- **Entropy:** `-Σpᵢ log₂pᵢ` — more sensitive to class probabilities

In most cases, the difference in final accuracy is small. This section explores whether one outperforms the other on the Iris dataset.

In [ ]:
results = []

for criterion in ['gini', 'entropy']:
    for depth in range(1, 12):
        m = DecisionTreeClassifier(
            criterion=criterion,
            max_depth=depth,
            random_state=RANDOM_STATE
        )
        m.fit(X_train, y_train)
        results.append({
            'criterion' : criterion.capitalize(),
            'max_depth' : depth,
            'train_acc' : accuracy_score(y_train, m.predict(X_train)),
            'test_acc'  : accuracy_score(y_test,  m.predict(X_test)),
            'n_leaves'  : m.get_n_leaves()
        })

results_df = pd.DataFrame(results)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Gini vs. Entropy — Performance Across Depths', fontsize=14, fontweight='bold')

palette = {'Gini': '#4CAF50', 'Entropy': '#2196F3'}

# Test accuracy
for criterion, color in palette.items():
    subset = results_df[results_df['criterion'] == criterion]
    axes[0].plot(subset['max_depth'], subset['test_acc'], 'o-',
                 label=criterion, color=color, linewidth=2, markersize=6)

axes[0].set_title('Test Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].set_ylim(0.7, 1.05)
axes[0].set_xticks(range(1, 12))

# Leaf count
for criterion, color in palette.items():
    subset = results_df[results_df['criterion'] == criterion]
    axes[1].plot(subset['max_depth'], subset['n_leaves'], 's-',
                 label=criterion, color=color, linewidth=2, markersize=6)

axes[1].set_title('Number of Leaves (Tree Complexity)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Leaf Count')
axes[1].legend()
axes[1].set_xticks(range(1, 12))

plt.tight_layout()
plt.savefig('gini_vs_entropy.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Head-to-head comparison at depth = 4
print("Head-to-Head Comparison at max_depth = 4:")
print("-" * 55)
print(f"  {'Metric':<30} {'Gini':>12} {'Entropy':>12}")
print("-" * 55)

for criterion in ['gini', 'entropy']:
    m = DecisionTreeClassifier(criterion=criterion, max_depth=4, random_state=RANDOM_STATE)
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    if criterion == 'gini':
        g_acc   = accuracy_score(y_test, preds)
        g_f1    = f1_score(y_test, preds, average='macro')
        g_depth = m.get_depth()
        g_leaves = m.get_n_leaves()
    else:
        e_acc   = accuracy_score(y_test, preds)
        e_f1    = f1_score(y_test, preds, average='macro')
        e_depth = m.get_depth()
        e_leaves = m.get_n_leaves()

print(f"  {'Test Accuracy':<30} {g_acc:>12.4f} {e_acc:>12.4f}")
print(f"  {'F1 Score (Macro)':<30} {g_f1:>12.4f} {e_f1:>12.4f}")
print(f"  {'Actual Depth':<30} {g_depth:>12} {e_depth:>12}")
print(f"  {'Number of Leaves':<30} {g_leaves:>12} {e_leaves:>12}")

print("\n💡 Both criteria produce similar accuracy on this dataset.")
print("   Gini is preferred in practice due to computational efficiency.")
print("   Entropy may produce slightly different tree structures.")

---

## 19. Practical Observations

A summary of key insights discovered during this analysis.

In [ ]:
# ── Comprehensive Results Summary ────────────────────────────────
print("=" * 65)
print("  COMPREHENSIVE RESULTS SUMMARY")
print("=" * 65)

print("\n📊 Dataset:")
print(f"   Samples: {len(X)} | Features: {X.shape[1]} | Classes: {len(class_names)}")
print(f"   Train: {len(X_train)} | Test: {len(X_test)}")

print("\n📈 Model Performance (Default, max_depth=4, Gini):")
print(f"   Train Accuracy       : {train_acc:.4f}")
print(f"   Test Accuracy        : {test_acc:.4f}")
print(f"   Macro F1 Score       : {f1_macro:.4f}")
print(f"   5-Fold CV Mean Acc   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print("\n🔍 Feature Importance (Top 2):")
for _, row in feat_imp_df.head(2).iterrows():
    print(f"   {row['Feature']:<35} {row['Importance']:.4f}")

print("\n⚙️ Best Hyperparameters (GridSearchCV):")
for k, v in best_params.items():
    print(f"   {k:<25} : {v}")
print(f"   Best CV Accuracy     : {best_cv_score:.4f}")

print("\n🌳 Tree Structure (Default):")
print(f"   Depth                : {clf.get_depth()}")
print(f"   Total Nodes          : {clf.tree_.node_count}")
print(f"   Leaf Nodes           : {clf.get_n_leaves()}")

print("\n💡 Key Observations:")
observations = [
    "Petal Length is the single most discriminative feature (high importance).",
    "Setosa is linearly separable from other classes — visible in pairplot.",
    "Versicolor and Virginica overlap, requiring deeper splits.",
    "max_depth=4 offers an excellent balance between accuracy and generalization.",
    "Gini and Entropy produce near-identical accuracy on this dataset.",
    "Decision Trees need no feature scaling — a key practical advantage.",
    "The model is highly interpretable — each decision path is human-readable."
]
for i, obs in enumerate(observations, 1):
    print(f"   {i}. {obs}")

In [ ]:
# ── Final Dashboard Plot ─────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Decision Tree — Complete Results Dashboard', fontsize=16, fontweight='bold', y=1.01)

# 1. Accuracy by depth
ax1 = fig.add_subplot(2, 3, 1)
ax1.plot(depths, train_accs, 'o-', color='#E53935', label='Train', linewidth=2)
ax1.plot(depths, test_accs,  's-', color='#1E88E5', label='Test',  linewidth=2)
ax1.axvline(x=4, color='gray', linestyle='--', alpha=0.6)
ax1.set_title('Accuracy vs Depth', fontweight='bold')
ax1.set_xlabel('max_depth'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.set_ylim(0.7, 1.05)

# 2. Confusion matrix heatmap
ax2 = fig.add_subplot(2, 3, 2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=class_names, yticklabels=class_names, cbar=False)
ax2.set_title('Confusion Matrix', fontweight='bold')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Actual')

# 3. Feature importance
ax3 = fig.add_subplot(2, 3, 3)
colors_bar = ['#E53935' if i == 0 else '#90CAF9' for i in range(len(feat_imp_df))]
ax3.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color=colors_bar)
ax3.set_title('Feature Importance', fontweight='bold')
ax3.set_xlabel('Importance Score')
ax3.invert_yaxis()

# 4. Metrics bar chart
ax4 = fig.add_subplot(2, 3, 4)
x = np.arange(len(class_names))
w = 0.25
ax4.bar(x - w, precision_per_class, w, label='Precision', color='#4CAF50', alpha=0.85)
ax4.bar(x,     recall_per_class,    w, label='Recall',    color='#2196F3', alpha=0.85)
ax4.bar(x + w, f1_per_class,        w, label='F1 Score',  color='#FF9800', alpha=0.85)
ax4.set_xticks(x); ax4.set_xticklabels(class_names)
ax4.set_title('Metrics by Class', fontweight='bold')
ax4.set_ylabel('Score'); ax4.set_ylim(0, 1.2)
ax4.legend(fontsize=8)

# 5. CV scores distribution
ax5 = fig.add_subplot(2, 3, 5)
ax5.bar(range(1, 6), cv_scores, color='#7B1FA2', alpha=0.8, edgecolor='white')
ax5.axhline(y=cv_scores.mean(), color='orange', linestyle='--', linewidth=2, label=f'Mean: {cv_scores.mean():.4f}')
ax5.set_title('5-Fold CV Accuracy', fontweight='bold')
ax5.set_xlabel('Fold'); ax5.set_ylabel('Accuracy')
ax5.set_ylim(0.85, 1.05); ax5.legend(fontsize=9)

# 6. Gini vs Entropy comparison
ax6 = fig.add_subplot(2, 3, 6)
gini_res    = results_df[results_df['criterion'] == 'Gini']
entropy_res = results_df[results_df['criterion'] == 'Entropy']
ax6.plot(gini_res['max_depth'],    gini_res['test_acc'],    'o-', color='#4CAF50', label='Gini',    linewidth=2)
ax6.plot(entropy_res['max_depth'], entropy_res['test_acc'], 's-', color='#F44336', label='Entropy', linewidth=2)
ax6.set_title('Gini vs Entropy (Test)', fontweight='bold')
ax6.set_xlabel('max_depth'); ax6.set_ylabel('Test Accuracy')
ax6.legend(); ax6.set_ylim(0.7, 1.05)

plt.tight_layout()
plt.savefig('results_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()

---

## 20. Conclusion

### What We Accomplished

In this notebook, we built a complete Decision Tree pipeline:

| Step | What We Did |
|------|-------------|
| **EDA** | Verified clean data, balanced classes, identified petal features as strong predictors |
| **Visualization** | Pairplot, boxplot, heatmap revealed separability and correlations |
| **Modeling** | Trained CART classifier with Gini criterion, max_depth=4 |
| **Evaluation** | Measured accuracy, confusion matrix, precision, recall, F1 |
| **Visualization** | Plotted the tree — fully interpretable decision rules |
| **Feature Importance** | Petal Length is the strongest predictor |
| **Hyperparameter Tuning** | GridSearchCV identified optimal depth and leaf settings |
| **Gini vs Entropy** | Both produce comparable accuracy; Gini is computationally faster |

### Final Model Metrics

| Metric | Score |
|--------|-------|
| Test Accuracy | See output above |
| Macro F1 Score | See output above |
| 5-Fold CV Accuracy | See output above |

### Key Takeaways

1. **Decision Trees are powerful AND interpretable** — every prediction has a human-readable explanation.
2. **Feature importance** gives actionable insights for domain experts.
3. **Controlling max_depth** is the single most effective way to prevent overfitting.
4. **Gini and Entropy** produce similar results; choose Gini for speed in practice.
5. **Decision Trees are the building block** of powerful ensemble methods like Random Forest and XGBoost — understanding them deeply unlocks all of modern ML.

### Next Steps

- 🌲 **Random Forest:** Train 100+ trees and aggregate predictions to reduce variance
- ⚡ **XGBoost:** Sequential tree building with gradient boosting for higher accuracy
- 🔪 **Cost-Complexity Pruning:** Use `ccp_alpha` with cross-validation
- 📊 **ROC/AUC Curves:** Evaluate probability calibration across thresholds
- 🏥 **Apply to a real dataset:** Try on healthcare, finance, or customer data

In [ ]:
print("=" * 60)
print("  NOTEBOOK COMPLETE")
print("=" * 60)
print()
print("  Files generated:")
print("  ✅ feature_distributions.png")
print("  ✅ pairplot.png")
print("  ✅ correlation_heatmap.png")
print("  ✅ boxplot.png")
print("  ✅ confusion_matrix.png")
print("  ✅ metrics_barplot.png")
print("  ✅ decision_tree.png")
print("  ✅ feature_importance.png")
print("  ✅ depth_vs_accuracy.png")
print("  ✅ gini_vs_entropy.png")
print("  ✅ results_dashboard.png")
print()
print("  Happy Learning! 🌳")